# Straight-line versus quadratic-Bezier comparison runner — validation only

This notebook clones the public repository without a token, pins exact runner source commit `398a2bfb7bd65ed8b4bbc93fb8cc05564f7f3c1b`, runs the complete test suite, validates the frozen target manifest and fail-closed runner, and checks that an unauthorized execution attempt creates no output. It does not run the comparison, generate paintings, train a model, mount Drive, require a GPU, or modify a closed experiment. Run all cells in order on a standard Colab CPU runtime.

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path('/content/latent-stroke-dynamics')
REPO_URL = 'https://github.com/Navid111/latent-stroke-dynamics.git'
BRANCH = 'quadratic-bezier-extension'
EXPECTED_COMMIT = '398a2bfb7bd65ed8b4bbc93fb8cc05564f7f3c1b'

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
completed = subprocess.run(
    ['git', 'clone', '--quiet', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
    capture_output=True,
    text=True,
)
if completed.returncode != 0:
    print(completed.stdout)
    print(completed.stderr)
    raise RuntimeError('Public repository clone failed.')
subprocess.run(['git', 'checkout', '--quiet', '--detach', EXPECTED_COMMIT], cwd=REPO_DIR, check=True)
observed_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert observed_commit == EXPECTED_COMMIT, observed_commit
assert subprocess.check_output(['git', 'status', '--short'], cwd=REPO_DIR, text=True).strip() == ''
print('CELL 1 COMPLETE — PUBLIC REPOSITORY CLONED; EXACT RUNNER SOURCE PINNED', observed_commit)

In [ ]:
import subprocess
from pathlib import Path

subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO_DIR, check=True)
test_run = subprocess.run(
    ['python', '-m', 'pytest', '-q'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
PYTEST_LOG = Path('/content/quadratic_bezier_runner_pytest.txt')
PYTEST_LOG.write_text(test_run.stdout + test_run.stderr, encoding='utf-8')
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
assert test_run.returncode == 0, 'The complete test suite failed.'
print('CELL 2 COMPLETE — COMPLETE TEST SUITE PASSED')

In [ ]:
import json
import subprocess
from pathlib import Path

validation_run = subprocess.run(
    ['python', 'run_quadratic_bezier_comparison.py', '--validate-only'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
if validation_run.returncode != 0:
    print(validation_run.stdout)
    print(validation_run.stderr)
    raise RuntimeError('Runner validation-only command failed.')
VALIDATION_JSON = Path('/content/quadratic_bezier_runner_validation.json')
VALIDATION_JSON.write_text(validation_run.stdout, encoding='utf-8')
validation = json.loads(validation_run.stdout)
print('CELL 3 COMPLETE — RUNNER VALIDATION REPORT CREATED IN /content ONLY')

In [ ]:
import subprocess
from pathlib import Path

UNAUTHORIZED_OUTPUT = Path('/content/quadratic-bezier-unauthorized-probe')
UNAUTHORIZED_INCOMPLETE = Path(str(UNAUTHORIZED_OUTPUT) + '.incomplete')
assert not UNAUTHORIZED_OUTPUT.exists()
assert not UNAUTHORIZED_INCOMPLETE.exists()
unauthorized = subprocess.run(
    [
        'python',
        'run_quadratic_bezier_comparison.py',
        '--execute',
        '--authorization',
        'configs/quadratic-bezier-target-freeze-2026-09-04.json',
        '--output-dir',
        str(UNAUTHORIZED_OUTPUT),
        '--source-commit',
        EXPECTED_COMMIT,
    ],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
UNAUTHORIZED_LOG = Path('/content/quadratic_bezier_unauthorized_probe.txt')
UNAUTHORIZED_LOG.write_text(unauthorized.stdout + unauthorized.stderr, encoding='utf-8')
assert unauthorized.returncode != 0, 'Unauthorized execution unexpectedly succeeded.'
assert not UNAUTHORIZED_OUTPUT.exists(), 'Unauthorized execution created a completed output.'
assert not UNAUTHORIZED_INCOMPLETE.exists(), 'Unauthorized execution created an incomplete output.'
print('CELL 4 COMPLETE — UNAUTHORIZED EXECUTION REFUSED WITH ZERO OUTPUT')

In [ ]:
import subprocess

assert validation['status'] == 'quadratic_bezier_comparison_runner_valid_no_outputs'
assert validation['protocol_id'] == 'quadratic_bezier_extension_v1'
assert validation['target_set_sha256'] == '26bada941bfd8f49f09333d70d397364e82f5ddbb6e1228324f24fb9d2b30bfd'
assert validation['target_count'] == 6
assert validation['seed_count'] == 3
assert validation['condition_count'] == 2
assert validation['expected_pair_count'] == 18
assert validation['expected_run_count'] == 36
assert validation['output_side_effects'] is False
assert validation['execution_authorized'] is False
assert validation['comparative_outputs_viewed'] is False
assert validation['training_performed'] is False
assert validation['learned_model_used'] is False
assert validation['closed_experiments_changed'] is False
assert set(validation['synthetic_smoke']) == {'straight', 'quadratic_bezier'}
assert all(item['deterministic'] for item in validation['synthetic_smoke'].values())
assert all(item['monotonic'] for item in validation['synthetic_smoke'].values())
worktree_status = subprocess.check_output(['git', 'status', '--short'], cwd=REPO_DIR, text=True).strip()
assert worktree_status == '', worktree_status
print('CELL 5 COMPLETE — FAIL-CLOSED RUNNER VALIDATION PASSED')
print('status:', validation['status'])
print('target-freeze SHA-256:', validation['target_freeze_sha256'])
print('expected runs:', validation['expected_run_count'])
print('environment:', validation['environment'])
print('straight smoke:', validation['synthetic_smoke']['straight'])
print('quadratic smoke:', validation['synthetic_smoke']['quadratic_bezier'])
print('unauthorized execution refused:', unauthorized.returncode != 0)
print('output side effects:', validation['output_side_effects'])
print('git status clean:', worktree_status == '')

In [ ]:
from google.colab import files

files.download(str(PYTEST_LOG))
files.download(str(VALIDATION_JSON))
files.download(str(UNAUTHORIZED_LOG))
print('CELL 6 COMPLETE — THREE VALIDATION DOWNLOADS STARTED')